# Resumes — Binoculars Scoring (canonical script)

This is the exact script that produced the resume numbers in §8 of the report (33.23% on humans, 10.13% on LLM-generated). It is the only place in the project where the resume-specific 4-bit NF4 quantization + CPU spillover are wired.

**Run environment:** Google Colab with a T4-class GPU (12 GiB VRAM target + 11 GiB CPU spillover). Mount Drive before running.

**Inputs / outputs (relative to working dir, typically `/content/`):**
- Input: `binoculars_eval_data_FINAL.csv` (4,537 rows; columns `Resume_ID, Text_Block, Is_AI, Dataset`) — copy from the project Drive's `resumes/` folder.
- Output: `binoculars_FINAL_RESULTS.csv` with added `Binoculars_Score`, `Binoculars_Prediction`, `Predicted_Is_AI`.
- Partial checkpoint: `binoculars_FULL_results_partial.csv` (refreshed every 100 rows; safe to resume from).

**Setup:**
```
git clone https://github.com/ahans30/Binoculars.git /content/Binoculars
pip install -r /content/Binoculars/requirements.txt
```

**Notes vs. `modularBinoculars/main.py`:** this script applies a per-row 2000-character truncation and scores one row at a time (no batching) — both are resume-specific accommodations for the T4 memory budget. The threshold (`0.9015310749276843`) is consumed inside `bino.predict()`, so AI/human predictions match the rest of the project.


In [ ]:
import os
import sys
import gc
import time
import pandas as pd
import torch
import transformers
from transformers import BitsAndBytesConfig

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

if '/content/Binoculars' not in sys.path:
    sys.path.append('/content/Binoculars')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

if not getattr(transformers.AutoModelForCausalLM.from_pretrained, "_is_patched", False):
    original_from_pretrained = transformers.AutoModelForCausalLM.from_pretrained
    original_tokenizer = transformers.AutoTokenizer.from_pretrained

    @classmethod
    def patched_from_pretrained(cls, pretrained_model_name_or_path, *model_args, **kwargs):
        gc.collect()
        torch.cuda.empty_cache()

        kwargs['quantization_config'] = bnb_config
        kwargs['device_map'] = 'auto'
        kwargs['torch_dtype'] = torch.float16
        kwargs['max_memory'] = {0: "12GiB", "cpu": "11GiB"}
        kwargs['trust_remote_code'] = False
        kwargs.pop('load_in_4bit', None)

        return original_from_pretrained.__func__(cls, pretrained_model_name_or_path, *model_args, **kwargs)

    @classmethod
    def patched_tokenizer(cls, pretrained_model_name_or_path, *model_args, **kwargs):
        kwargs['trust_remote_code'] = False
        return original_tokenizer.__func__(cls, pretrained_model_name_or_path, *model_args, **kwargs)

    patched_from_pretrained._is_patched = True
    patched_tokenizer._is_patched = True

    transformers.AutoModelForCausalLM.from_pretrained = patched_from_pretrained
    transformers.AutoTokenizer.from_pretrained = patched_tokenizer


from binoculars import Binoculars

print("\nLoading Falcon-7B Models... (Using Native Implementation & CPU Spillover)")
bino = Binoculars()
print("Models loaded successfully!\n")

df = pd.read_csv('binoculars_eval_data_FINAL.csv')

print(f"Running detection on {len(df)} total resumes. This may take 1-2 hours.")
print("Auto-saving progress every 100 resumes to 'binoculars_FULL_results_partial.csv'...")

predictions = []
scores = []

start_time = time.time()
for index, text in enumerate(df['Text_Block']):
    safe_text = str(text)[:2000]

    score = bino.compute_score(safe_text)
    pred = bino.predict(safe_text)

    scores.append(score)
    predictions.append(pred)

    # show every 50 resumes
    if (index + 1) % 50 == 0:
        elapsed = round((time.time() - start_time) / 60, 2)
        print(f"Processed {index + 1}/{len(df)} resumes... (Elapsed time: {elapsed} minutes)")

    # save data from every 100 resumes
    if (index + 1) % 100 == 0:
        temp_df = df.iloc[:index+1].copy()
        temp_df['Binoculars_Score'] = scores
        temp_df['Binoculars_Prediction'] = predictions
        temp_df['Predicted_Is_AI'] = temp_df['Binoculars_Prediction'].apply(lambda x: 1 if "AI" in str(x) else 0)
        temp_df.to_csv('binoculars_FULL_results_partial.csv', index=False)

end_time = time.time()

# finalize df
df['Binoculars_Score'] = scores
df['Binoculars_Prediction'] = predictions
df['Predicted_Is_AI'] = df['Binoculars_Prediction'].apply(lambda x: 1 if "AI" in str(x) else 0)

# accuracy calc
correct_predictions = (df['Is_AI'] == df['Predicted_Is_AI']).sum()
accuracy = (correct_predictions / len(df)) * 100

print(f"\n--- FULL EVALUATION COMPLETE in {round((end_time - start_time) / 60, 2)} minutes ---")
print(f"Overall Accuracy on the entire dataset: {accuracy:.2f}%\n")


output_file = 'binoculars_FINAL_RESULTS.csv'
df.to_csv(output_file, index=False)
print(f"Saved final results to '{output_file}'!")